# Tutorial: BOLD From Whole-Brain AdEx Simulations (Maria Sacha Parameterisation)

This notebook uses **real whole-brain TVBToolkit simulations** (not toy data), following the anaesthesia-oriented setup used in the original TVBSim BOLD tutorial.

Pipeline:
1. Simulate AdEx/Zerlaut whole-brain conditions (awake, sleep-like, ketamine-like, propofol-like).
2. Extract mean-field firing rates.
3. Transform rates to BOLD with the TVB-style first-order Volterra HRF.
4. Compute FC-SC coupling from preprocessed BOLD.
5. Compare brain-state analysis on rates (pre-BOLD) vs BOLD (post-transform).

## Equations (TVB first-order Volterra HRF)

For neural drive $x(t)$ and HRF kernel $G(t)$:

\[
B(t) = (x * G)(t)
\]

with

\[
G(t)=\frac{1}{3}\exp\left(-\frac{t}{2\tau_s}\right)
\frac{\sin\left(\sqrt{\frac{1}{\tau_f}-\frac{1}{4\tau_s^2}}\,t\right)}
{\sqrt{\frac{1}{\tau_f}-\frac{1}{4\tau_s^2}}},
\]

using defaults $\tau_s=0.8$, $\tau_f=0.4$, scaling $(\cdot -1)k_1V_0$ with $k_1=5.6$, $V_0=0.02$.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

from tvbtoolkit import (
    WholeBrainConfig,
    OutputConfig,
    ConditionSpec,
    run_condition_batch,
)
from tvbtoolkit.bold import (
    BOLDParams,
    bold_from_firing_rates,
    corr_fc_sc,
    preprocess_bold_signal,
)
from tvbtoolkit.analysis import summarize_brain_states
from tvbtoolkit.visualization import set_publication_style

set_publication_style()

## 1) Simulation setup (legacy-style anaesthetic parameterisation)

Values below mirror the TVBSim tutorial and associated Maria Sacha parameter set.

In [ ]:
RUN_CFG = {
    # Tutorial defaults keep runtime reasonable while making FC-SC less unstable.
    "simulation_length_ms": 30000.0,
    "cut_transient_ms": 5000.0,
    "dt_ms": 0.1,
    "monitor_mode": "temporal_average",
    "temporal_average_period_ms": 1.0,
    "tr_ms": 500.0,  # denser in-silico BOLD sampling for robust FC estimation
    "seeds": [0, 1, 2],
    "n_jobs": 1,
    "use_processes": False,
}

out = OutputConfig(root=Path("outputs") / "tutorial_bold_from_rates")
fig_dir = out.figures_dir / "bold_tutorial"
fig_dir.mkdir(parents=True, exist_ok=True)
print("Outputs:", out.root.resolve())
print("Figures:", fig_dir.resolve())
print("Note: for publication inference, increase simulation_length_ms and number of seeds further.")

In [ ]:
# Maria Sacha / TVBSim tutorial transfer-function parameterisation
P_e = [-0.05017034, 0.00451531, -0.00794377, -0.00208418, -0.00054697,
       0.00194753, 0.00274079, 0.00341614, -0.01066769, -0.01156433]
P_i = [-0.05184978, 0.0061593, -0.01403522, 0.00166511, -0.0020559,
       0.00656668, 0.00171829, 0.00318432, -0.04516385, -0.03112775]

base_cfg = WholeBrainConfig(
    model_family="adex_zerlaut",
    zerlaut_matteo=False,
    zerlaut_gk_gna=False,
    zerlaut_order=1,
    stochastic_integrator=True,
    simulation_length_ms=RUN_CFG["simulation_length_ms"],
    dt_ms=RUN_CFG["dt_ms"],
    conduction_speed=4.0,
    coupling_strength=0.3,
    monitor_mode=RUN_CFG["monitor_mode"],
    temporal_average_period_ms=RUN_CFG["temporal_average_period_ms"],
    parameter_overrides={
        "parameter_model": {
            "E_L_e": -64.0,
            "E_L_i": -65.0,
            "P_e": P_e,
            "P_i": P_i,
        },
    },
)

conditions = [
    ConditionSpec(
        name="awake",
        description="Awake baseline",
        parameter_overrides={"parameter_model": {"b_e": 5.0}},
    ),
    ConditionSpec(
        name="sleep_like",
        description="Sleep-like high adaptation",
        parameter_overrides={"parameter_model": {"b_e": 120.0}},
    ),
    ConditionSpec(
        name="ketamine_like",
        description="Ketamine-like adaptation and excitatory/inhibitory synaptic time constants",
        parameter_overrides={"parameter_model": {"b_e": 30.0, "tau_e_e": 3.75, "tau_e_i": 3.75}},
    ),
    ConditionSpec(
        name="propofol_like",
        description="Propofol-like adaptation and inhibitory time constant",
        parameter_overrides={"parameter_model": {"b_e": 30.0, "tau_i": 7.0}},
    ),
]

conditions

In [ ]:
metrics_by_condition = run_condition_batch(
    base_cfg=base_cfg,
    conditions=conditions,
    seeds=RUN_CFG["seeds"],
    output=out,
    post_stim_window=300,
    save_timeseries=True,
    n_jobs=RUN_CFG["n_jobs"],
    use_processes=RUN_CFG["use_processes"],
    show_progress=True,
    monitor_mode_default="temporal_average",
    temporal_average_period_ms=RUN_CFG["temporal_average_period_ms"],
)

list(metrics_by_condition.keys())

## 2) Extract firing rates after transient removal

In [ ]:
rates_by_condition = {}
time_by_condition_s = {}

for cond, m in metrics_by_condition.items():
    t_ms = np.asarray(m["time_ms_example"], dtype=float)
    x_khz = np.asarray(m["raw_example"], dtype=float)  # TVB state variable in kHz

    keep = t_ms >= RUN_CFG["cut_transient_ms"]
    t_ms = t_ms[keep]
    x_khz = x_khz[keep]

    rates_by_condition[cond] = x_khz
    time_by_condition_s[cond] = t_ms / 1000.0

{k: v.shape for k, v in rates_by_condition.items()}

In [ ]:
# Figure 1: stacked firing rates (Hz), one panel per condition
n_cond = len(rates_by_condition)
fig, axes = plt.subplots(n_cond, 1, figsize=(13, 2.6 * n_cond), sharex=True)
if n_cond == 1:
    axes = [axes]

for ax, (cond, x_khz) in zip(axes, rates_by_condition.items()):
    x_hz = 1000.0 * x_khz
    t_s = time_by_condition_s[cond]
    n_regions = min(18, x_hz.shape[1])
    step = np.percentile(x_hz[:, :n_regions], 99) * 1.05
    offsets = np.arange(n_regions) * step
    for r in range(n_regions):
        ax.plot(t_s, x_hz[:, r] + offsets[r], lw=0.8, alpha=0.85)
    ax.set_ylabel("Offset + Hz")
    ax.set_title(f"{cond.replace('_', ' ').title()} rates (stacked {n_regions} regions)")
    ax.grid(alpha=0.22)

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
fig.savefig(fig_dir / "01_stacked_rates_real_sim.svg", bbox_inches="tight")
fig.savefig(fig_dir / "01_stacked_rates_real_sim.pdf", bbox_inches="tight")
plt.show()

## 3) Convert rates to BOLD and visualise

In [ ]:
bold_by_condition = {}
bold_time_by_condition_s = {}

for cond, x_khz in rates_by_condition.items():
    t_ms = time_by_condition_s[cond] * 1000.0
    if t_ms.size > 1:
        dt_ms = float(np.median(np.diff(t_ms)))
    else:
        dt_ms = 1.0

    out_bold = bold_from_firing_rates(
        x_khz,
        dt_ms=dt_ms,
        tr_ms=RUN_CFG["tr_ms"],
        hrf_length_ms=20000.0,
        return_debug=True,
    )

    bold_by_condition[cond] = out_bold["bold"]
    bold_time_by_condition_s[cond] = out_bold["time_ms"] / 1000.0

{k: v.shape for k, v in bold_by_condition.items()}

In [ ]:
# Figure 2: stacked BOLD traces, one panel per condition
n_cond = len(bold_by_condition)
fig, axes = plt.subplots(n_cond, 1, figsize=(13, 2.6 * n_cond), sharex=True)
if n_cond == 1:
    axes = [axes]

for ax, (cond, b) in zip(axes, bold_by_condition.items()):
    t_s = bold_time_by_condition_s[cond]
    n_regions = min(18, b.shape[1])
    step = np.percentile(np.abs(b[:, :n_regions]), 99) * 1.8 + 1e-9
    offsets = np.arange(n_regions) * step
    for r in range(n_regions):
        ax.plot(t_s, b[:, r] + offsets[r], lw=1.0, alpha=0.9)
    ax.set_ylabel("Offset + BOLD")
    ax.set_title(f"{cond.replace('_', ' ').title()} BOLD (stacked {n_regions} regions)")
    ax.grid(alpha=0.22)

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
fig.savefig(fig_dir / "02_stacked_bold_real_sim.svg", bbox_inches="tight")
fig.savefig(fig_dir / "02_stacked_bold_real_sim.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3: one-region overlay (rate vs BOLD) for awake condition
cond_ref = "awake"
region_idx = 0

rate_hz = 1000.0 * rates_by_condition[cond_ref][:, region_idx]
rate_z = (rate_hz - rate_hz.mean()) / (rate_hz.std() + 1e-12)

bold_ref = bold_by_condition[cond_ref][:, region_idx]
bold_z = (bold_ref - bold_ref.mean()) / (bold_ref.std() + 1e-12)

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(time_by_condition_s[cond_ref], rate_z, lw=1.2, label="Rate (z-score)")
ax.plot(bold_time_by_condition_s[cond_ref], bold_z, lw=2.2, label="BOLD (z-score)")
ax.set_title("Awake condition, region 1: neural drive vs BOLD")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalised amplitude")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(fig_dir / "03_overlay_awake_rate_vs_bold.svg", bbox_inches="tight")
fig.savefig(fig_dir / "03_overlay_awake_rate_vs_bold.pdf", bbox_inches="tight")
plt.show()

## 4) FC-SC coupling from BOLD (legacy-style preprocessing)

This section now computes FC-SC **per seed** from saved simulation outputs, then summarises by condition.

It reports two coupling metrics:
- `legacy_r_signed_full`: TVBSim-style signed FC vs SC over flattened matrices
- `masked_r_abs_upper`: upper-triangle correlation between `|FC|` and SC on non-zero SC edges

The second metric is often more interpretable when SC is sparse/zero-masked.

In [ ]:
from tvb.simulator import lab

conn_zip = Path("..") / "data" / "connectivity" / "connectivity_68.zip"
conn = lab.connectivity.Connectivity().from_file(str(conn_zip.resolve()))
sc = np.asarray(conn.weights, dtype=float)
np.fill_diagonal(sc, 0.0)

iu = np.triu_indices_from(sc, k=1)
mask_nonzero_sc = sc[iu] > 0

def _safe_corr(a, b):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    if a.size != b.size or a.size < 3:
        return np.nan
    if (np.std(a) <= 0) or (np.std(b) <= 0):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

fcsc_seedwise = {}
fc_mean_for_plot = {}

for cond in metrics_by_condition.keys():
    cond_dir = out.simulations_dir / cond
    seed_files = []
    for s in RUN_CFG["seeds"]:
        sf = cond_dir / f"seed_{int(s):03d}.npz"
        if sf.exists():
            seed_files.append(sf)
    if not seed_files:
        continue

    rows = []
    fcs = []
    for sf in seed_files:
        d = np.load(sf, allow_pickle=True)
        t_ms = np.asarray(d["time_ms"], dtype=float)
        raw = np.asarray(d["raw"], dtype=float)

        keep = t_ms >= RUN_CFG["cut_transient_ms"]
        t_ms = t_ms[keep]
        raw = raw[keep]
        if t_ms.size < 30:
            continue

        dt_ms = float(np.median(np.diff(t_ms))) if t_ms.size > 1 else 1.0
        b = bold_from_firing_rates(raw, dt_ms=dt_ms, tr_ms=RUN_CFG["tr_ms"])

        bp = BOLDParams(TR=RUN_CFG["tr_ms"] / 1000.0, n_order=2, low_f_num=0.01, high_f_num=0.1)
        b_pp = preprocess_bold_signal(b, params=bp, apply_zscore=True, apply_bandpass=True)

        fc_abs, legacy_r = corr_fc_sc(b_pp, sc)
        fc_signed = np.corrcoef(b_pp.T)
        masked_r_abs_upper = _safe_corr(fc_abs[iu][mask_nonzero_sc], sc[iu][mask_nonzero_sc])
        masked_r_signed_upper = _safe_corr(fc_signed[iu][mask_nonzero_sc], sc[iu][mask_nonzero_sc])

        rows.append((legacy_r, masked_r_abs_upper, masked_r_signed_upper, b_pp.shape[0]))
        fcs.append(fc_abs)

    if rows:
        arr = np.asarray(rows, dtype=float)
        fcsc_seedwise[cond] = {
            "legacy_r_signed_full": arr[:, 0],
            "masked_r_abs_upper": arr[:, 1],
            "masked_r_signed_upper": arr[:, 2],
            "n_bold_samples": arr[:, 3],
        }
        fc_mean_for_plot[cond] = np.mean(np.stack(fcs, axis=0), axis=0)

print("Per-condition FC-SC summary (mean ± SD):")
for cond, d in fcsc_seedwise.items():
    l = d["legacy_r_signed_full"]
    m = d["masked_r_abs_upper"]
    s = d["masked_r_signed_upper"]
    n = d["n_bold_samples"]
    print(
        f"- {cond}: "
        f"legacy={np.nanmean(l):.3f}±{np.nanstd(l):.3f}, "
        f"masked|FC|={np.nanmean(m):.3f}±{np.nanstd(m):.3f}, "
        f"masked signed FC={np.nanmean(s):.3f}±{np.nanstd(s):.3f}, "
        f"BOLD samples/seed ~{np.nanmean(n):.1f}"
    )

fig, axes = plt.subplots(2, len(fc_mean_for_plot), figsize=(4.0 * max(1, len(fc_mean_for_plot)), 6.8))
if len(fc_mean_for_plot) == 1:
    axes = np.array(axes).reshape(2, 1)

for j, (cond, fc_abs_mean) in enumerate(fc_mean_for_plot.items()):
    l_mean = np.nanmean(fcsc_seedwise[cond]["legacy_r_signed_full"])
    m_mean = np.nanmean(fcsc_seedwise[cond]["masked_r_abs_upper"])

    im0 = axes[0, j].imshow(fc_abs_mean, cmap="seismic", vmin=0.0, vmax=1.0, origin="lower")
    im1 = axes[1, j].imshow(sc, cmap="viridis", origin="lower")
    cond_title = cond.replace("_", " ").title()
    axes[0, j].set_title(f"{cond_title} |FC| mean\nlegacy r={l_mean:.3f}, masked |FC|-SC r={m_mean:.3f}")
    axes[1, j].set_title("SC")
    for a in (axes[0, j], axes[1, j]):
        a.set_xlabel("Region")
        a.set_ylabel("Region")

fig.colorbar(im0, ax=axes[0, :], fraction=0.02, pad=0.01)
fig.colorbar(im1, ax=axes[1, :], fraction=0.02, pad=0.01)
fig.tight_layout()
fig.savefig(fig_dir / "04_fc_sc_by_condition_seedwise.svg", bbox_inches="tight")
fig.savefig(fig_dir / "04_fc_sc_by_condition_seedwise.pdf", bbox_inches="tight")
plt.show()

## 5) Brain states before vs after BOLD (same TVBToolkit analysis method)

In [ ]:
cond_ref = "awake"

rates_ref = rates_by_condition[cond_ref]
bold_ref = bold_by_condition[cond_ref]

bs_rates = summarize_brain_states(rates_ref, n_states=5, random_seed=0, n_init=10)
bs_bold = summarize_brain_states(bold_ref, n_states=5, random_seed=0, n_init=10)

occ_r = bs_rates.occupancy
occ_b = bs_bold.occupancy

k = max(occ_r.size, occ_b.size)
occ_r = np.pad(occ_r, (0, k - occ_r.size))
occ_b = np.pad(occ_b, (0, k - occ_b.size))

x = np.arange(1, k + 1)
width = 0.36

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.bar(x - width / 2, occ_r, width=width, alpha=0.86, label="Rates")
ax.bar(x + width / 2, occ_b, width=width, alpha=0.86, label="BOLD")
ax.set_xlabel("State index (subject-local)")
ax.set_ylabel("Occupancy probability")
ax.set_title("Awake condition: brain-state occupancy before vs after BOLD")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(fig_dir / "05_occupancy_rates_vs_bold.svg", bbox_inches="tight")
fig.savefig(fig_dir / "05_occupancy_rates_vs_bold.pdf", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
im0 = axes[0].imshow(bs_rates.transition_matrix, cmap="magma", origin="lower")
im1 = axes[1].imshow(bs_bold.transition_matrix, cmap="magma", origin="lower")
axes[0].set_title("Transitions (rates)")
axes[1].set_title("Transitions (BOLD)")
for ax in axes:
    ax.set_xlabel("Next state")
    ax.set_ylabel("Current state")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(fig_dir / "06_transitions_rates_vs_bold.svg", bbox_inches="tight")
fig.savefig(fig_dir / "06_transitions_rates_vs_bold.pdf", bbox_inches="tight")
plt.show()

## Notes

- The anaesthetic conditions and transfer-function coefficients are taken from the legacy TVBSim tutorial setup.
- BOLD preprocessing safely handles short series via SciPy `method='gust'` fallback when needed.
- If condition ordering still looks unstable, increase: `simulation_length_ms`, `cut_transient_ms`, and number of seeds.
- In this notebook, FC-SC is now estimated per seed and summarised, rather than from one example trace.